# CONSTELLATION Tutorial: Compartment-Level LR Analysis of CRC Xenium Data

This notebook demonstrates **CONSTELLATION**'s compartment-level ligand–receptor (LR)
analysis on the [10x Xenium Human Colon Cancer (CRC) dataset](https://www.10xgenomics.com/datasets/xenium-v1-human-colon-cancer-p1-crc-add-on-ffpe).

## What is compartment-level analysis?

Instead of testing LR interactions per cell type (e.g., macrophage–T cell), compartment-level
analysis tests whether ligand–receptor spatial co-expression is enriched *within spatial
compartments* defined relative to the tumor boundary. This is useful when:

- Cell-type annotations are unavailable or unreliable
- You want to study how signaling changes with distance from an anatomical landmark
- The biological question is about *where* interactions occur, not *between whom*

CONSTELLATION uses the same **distance-weighted kernel test** (`T = dot(L, w)`) as the
cell-type level analysis, with edges restricted to within-compartment pairs. The analytical
null model (no permutations needed) tests whether the observed spatial coupling of ligand
and receptor expression exceeds what is expected under random cell labeling.

## Compartments

The CRC tissue has been pre-annotated into spatial compartments using marker-based tumor
detection and signed distance from the tumor boundary (see `tutorial_annotate_compartments.ipynb`):

| Compartment | Definition | Approx. cells |
|:---|:---|---:|
| **Tumor** | Inside the smoothed tumor boundary | ~93K |
| **50 micron** | Within 50 µm outside the tumor boundary (interface) | ~12K |
| **Tissue** | Normal tissue beyond 50 µm from the boundary | ~111K |

## Prerequisites

- `xenium_final.parquet` — cell metadata with compartment annotations (from `tutorial_annotate_compartments.ipynb`)
- `data/cell_feature_matrix.h5` — Xenium expression matrix (symlink or copy from raw data)
- `constellation` package installed (`pip install -e .` from the constellation repo)

In [ ]:
import numpy as np
import pandas as pd
import h5py
import scipy.sparse as sp
import matplotlib.pyplot as plt
import constellation as cst

np.random.seed(42)
print(f'CONSTELLATION version: {cst.__version__}')

In [ ]:
# ================================================================
# Paths
# ================================================================
DATA_DIR = 'data'                              # symlink to raw Xenium outputs
META_PATH = 'xenium_final.parquet'              # cell metadata with compartment labels
H5_PATH = f'{DATA_DIR}/cell_feature_matrix.h5'  # Xenium sparse expression matrix
OUTPUT_PATH = 'constellation_crc_results.csv'   # where to save results

# ================================================================
# Analysis parameters
# ================================================================
K = 10              # number of spatial nearest neighbors
                    # This defines the "neighborhood" for each cell.
                    # k=10 is a reasonable default for Xenium (median cell
                    # diameter ~6–7 µm); smaller k (5) tests tighter
                    # neighborhoods, larger k (20) averages over wider areas.

TAU = 7.0           # distance decay parameter (µm) for the exponential kernel
                    # K(d) = exp(-d/tau). Set to ~median NN1 distance (6.5 µm
                    # for this CRC dataset). Strongly weights direct contacts
                    # while still incorporating farther neighbors.

LR_RESOURCE = 'cellphonedb'  # LR database to use (see cst.show_lr_resources())

---
## 1. Load Data

We load two files:

1. **`xenium_final.parquet`** — cell metadata with spatial coordinates (`x_centroid`,
   `y_centroid`) and compartment labels (`compartment`). This was produced by
   `tutorial_annotate_compartments.ipynb`.

2. **`cell_feature_matrix.h5`** — the Xenium sparse gene expression matrix in 10x HDF5
   format (genes × cells, CSC). We subset to non-excluded cells and transpose to
   cells × genes.

Cells in the `Excluded` compartment (outside tissue mask or below the y=4000 µm cutoff)
are removed.

In [ ]:
# Load cell metadata
meta = pd.read_parquet(META_PATH)
print(f'Total cells: {len(meta):,}')

# Filter out 'Excluded' compartment
meta = meta[meta['compartment'] != 'Excluded'].reset_index(drop=True)
print(f'After filtering Excluded: {len(meta):,}')

# Load expression matrix from H5 (stored as CSC: genes x cells)
with h5py.File(H5_PATH, 'r') as f:
    data = f['matrix/data'][:]
    indices = f['matrix/indices'][:]
    indptr = f['matrix/indptr'][:]
    shape = f['matrix/shape'][:]
    genes = [g.decode() for g in f['matrix/features/name'][:]]
    feature_types = [t.decode() for t in f['matrix/features/feature_type'][:]]
    barcodes = [b.decode() for b in f['matrix/barcodes'][:]]

# Build full matrix (genes x cells), then subset to non-excluded cells
mat_full = sp.csc_matrix((data, indices, indptr), shape=shape)
barcode_to_idx = {b: i for i, b in enumerate(barcodes)}
keep_idx = np.array([barcode_to_idx[cid] for cid in meta['cell_id']])

# Subset and transpose to cells x genes (CSR for fast row slicing)
mat = mat_full[:, keep_idx].T.tocsr()

# Keep only Gene Expression features (exclude controls)
gene_mask = np.array([ft == 'Gene Expression' for ft in feature_types])
genes = [g for g, m in zip(genes, gene_mask) if m]
mat = mat[:, gene_mask]
gene_to_idx = {g: i for i, g in enumerate(genes)}

print(f'Expression matrix: {mat.shape[0]:,} cells x {mat.shape[1]} genes')

# Extract coordinates and compartments
coords = meta[['x_centroid', 'y_centroid']].values
compartments = meta['compartment'].values
distance_values = meta['compartment_dist'].values

print(f'\nCells per compartment:')
for comp, n in meta['compartment'].value_counts().items():
    print(f'  {comp}: {n:,}')

---
## 2. Visualize Compartments

Before running the analysis, it is useful to visualize the spatial layout of the
compartments. This confirms that the tumor boundary, interface zone, and tissue region
are correctly defined and spatially contiguous.

CONSTELLATION provides `plot_compartment_spatial()` to produce a scatter plot of
cell positions colored by compartment.

In [ ]:
fig, ax = cst.plot_compartment_spatial(
    coords,
    compartments,
    compartment_colors={'Tumor': '#E07A5F', '50 micron': '#F2CC8F', 'Tissue': '#3D85C6'},
    title='CRC Xenium: Spatial Compartments',
    point_size=0.1,
)
ax.invert_yaxis()
plt.show()

---
## 3. LR Database

CONSTELLATION ships with several curated LR databases. Use `show_lr_resources()` to
list available options, and `load_lr_resource()` to load one.

We use **CellPhoneDB** (via LIANA), which contains ~1,200 curated ligand–receptor pairs.
After filtering to pairs where *both* genes are present in the Xenium panel, we obtain
the testable subset.

| Database | Pairs | Source |
|:---|---:|:---|
| `cellphonedb` | ~1,200 | CellPhoneDB v4 via LIANA |
| `celltalkdb` | ~3,300 | CellTalkDB (literature) |
| `connectomedb` | ~2,200 | Connectome DB 2020 |
| `consensus` | ~4,700 | LIANA consensus (union) |

To switch databases, change `LR_RESOURCE` above. Databases with more pairs yield more
testable interactions (and a stricter multiple-testing correction).

In [ ]:
# Show available LR resources
cst.show_lr_resources()

# Load CellPhoneDB
lr_df, lr_pairs_all = cst.load_lr_resource(LR_RESOURCE)
print(f'\nTotal LR pairs in {LR_RESOURCE}: {len(lr_pairs_all):,}')

# Filter to pairs testable in this Xenium panel
panel_genes = set(genes)
lr_pairs = cst.filter_lr_pairs_by_genes(lr_pairs_all, panel_genes)
print(f'Testable LR pairs (both genes in panel): {len(lr_pairs)}')
print()
for lig, rec in sorted(lr_pairs):
    print(f'  {lig} \u2192 {rec}')

---
## 4. Spatial Graph & Compartment Scan

CONSTELLATION needs a **spatial k-NN graph** to define which cells are neighbors of each
other. This is the core structure that determines the spatial scale of interaction testing.

### Choosing k

| k | NN1 distance | Interpretation |
|---:|---:|:---|
| 5 | ~6–7 µm | Tight: only immediately adjacent cells |
| **10** | **~6–7 µm** | **Default: captures direct-contact neighborhood** |
| 20 | ~6–7 µm | Broader: includes cells 2–3 diameters away |

The median nearest-neighbor distance (NN1) reflects the typical cell spacing. For Xenium
data with ~6–7 µm median cell diameter, k=10 captures the immediate contact neighborhood.

After building the graph, we run `scan_compartments()` to check how many LR pairs are
testable in each compartment (both ligand and receptor expressed above threshold).

In [ ]:
# Build spatial k-NN graph
knn_indices, knn_distances = cst.build_spatial_graph_from_coords(coords, k=K)
print(f'Median NN1 distance: {np.median(knn_distances[:, 0]):.2f} \u00b5m')

# Build expression dictionary for all LR genes
lr_genes = sorted(set(g for pair in lr_pairs for g in pair))
expr_dict = {}
for gene in lr_genes:
    idx = gene_to_idx[gene]
    expr_dict[gene] = np.asarray(mat[:, idx].todense()).ravel()
print(f'Expression vectors loaded for {len(expr_dict)} genes')

# Scan compartments for testable LR pairs
report = cst.scan_compartments(
    expr_dict=expr_dict,
    compartments=compartments,
    lr_pairs=lr_pairs,
    min_expr_frac=0.01,
    min_cells=10,
)

---
## 5. Run CONSTELLATION

For each (LR pair, compartment), CONSTELLATION tests whether ligand-receptor spatial
co-expression is enriched using the **distance-weighted kernel test**:

1. Edges are restricted to pairs where both cells belong to the same compartment
2. The test statistic is `T = dot(L, w)`, where `L = log1p(ligand expression)` and
   `w_i = sum_j K(d_ij) * R_j` with `K = exp(-d/tau)` and `R = log1p(receptor expression)`
3. Under the analytical permutation null (random relabeling of L), E[T] and Var[T] are
   computed in closed form (no permutations needed)

This is the same test used at the cell-type level, applied to spatial compartments.

**Output columns:**

| Column | Meaning |
|:---|:---|
| `fold_enrichment` | Observed T / expected T under null |
| `z_score` | Standard deviations above expected |
| `p_value` | Two-sided p-value from normal approximation |
| `p_adj` | BH-corrected FDR across all tests |
| `significant` | `True` if p_adj < 0.05 and z > 0 |

In [ ]:
import time

t0 = time.time()
results = cst.run_compartment_analysis(
    expr_dict=expr_dict,
    compartments=compartments,
    coords=coords,
    lr_pairs=lr_pairs,
    k=K,
    tau=TAU,
    indices=knn_indices,
    distances=knn_distances,
    min_cells=10,
    fdr_method='fdr_bh',
    fdr_threshold=0.05,
    verbose=True,
)
elapsed = time.time() - t0
print(f'\nCompleted in {elapsed:.1f}s')
print(f'Total tests: {len(results):,}')

sig = results[results['significant']]
print(f'Significant (FDR < 0.05, z > 0): {len(sig)}')

---
## 6. Results

### Interpreting the results table

- **fold_enrichment > 1** means the observed distance-weighted LR signal (`T = dot(L, w)`)
  exceeds the expected value under the permutation null. Values above 2 indicate strong
  spatial coupling; values below 1 indicate depletion.

- **z_score** reflects both effect size and statistical power. Large compartments
  (Tumor, Tissue) have more power, so even modest fold enrichments can reach high z.

- **Compartment differences**: An LR pair significant in Tumor but not Tissue suggests
  tumor-specific spatial signaling. Pairs significant in 50 micron highlight
  interface-enriched interactions.

### Key findings

- **CEACAM6-CEACAM5/CEACAM1**: strongest enrichment in Tissue (FE~7-9) — epithelial
  adhesion in normal colonic epithelium. Lower but significant in Tumor (FE~1.1-1.2).
- **CXCL13-CXCR5**: Tissue-enriched (FE=4.3) — tertiary lymphoid structures (TLS).
- **CXCL12-CXCR4**: Tumor-enriched (FE=2.5), depleted in Tissue (FE=0.76) — tumor
  microenvironment chemokine axis.
- **C3-C3AR1**: Tumor-enriched (FE=2.1) — complement activation in tumor.
- **CCL19-CCR7**: Tissue-enriched (FE=3.0) — lymphoid homing.

### Heatmap

The heatmap shows fold enrichment for each LR pair across compartments. Cells with
significant enrichment (FDR < 0.05) are marked with an asterisk.

In [ ]:
# Results summary
sig = results[results['significant']]

print('Significant LR pairs per compartment (FDR < 0.05, z > 0):')
for comp in ['Tumor', '50 micron', 'Tissue']:
    n_sig = sig[sig['compartment'] == comp].shape[0]
    n_total = results[results['compartment'] == comp].shape[0]
    print(f'  {comp}: {n_sig} / {n_total}')

# Top results by z-score
cols = ['lr_pair', 'compartment', 'fold_enrichment', 'z_score', 'p_adj']
print(f'\nTop 15 by z-score:')
display(sig.sort_values('z_score', ascending=False)[cols].head(15))

# Heatmap — use diverging colormap centered at 1.0 (no enrichment)
# so that fold < 1 (blue) and fold > 1 (red) are visually distinct
from matplotlib.colors import TwoSlopeNorm
compartment_order = ['Tumor', '50 micron', 'Tissue']

pivot = results.pivot_table(index='lr_pair', columns='compartment',
                            values='fold_enrichment', aggfunc='first')
pivot = pivot[compartment_order]

sig_pivot = results.pivot_table(index='lr_pair', columns='compartment',
                                values='significant', aggfunc='first')
sig_pivot = sig_pivot[compartment_order]

vmax = max(pivot.max().max(), 1.5)
norm = TwoSlopeNorm(vmin=0.8, vcenter=1.0, vmax=vmax)

fig, ax = plt.subplots(figsize=(5, max(6, len(pivot) * 0.4)))
im = ax.imshow(pivot.values, cmap='RdBu_r', norm=norm, aspect='auto')

ax.set_xticks(range(len(compartment_order)))
ax.set_xticklabels(compartment_order, fontsize=11)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=10)
ax.set_xlabel('Compartment', fontsize=12)
ax.set_ylabel('LR Pair', fontsize=12)
ax.set_title('CRC Compartment LR Analysis: Fold Enrichment', fontsize=13, fontweight='bold')

# Annotate: fold value + asterisk if significant
for i in range(len(pivot.index)):
    for j in range(len(compartment_order)):
        val = pivot.values[i, j]
        is_sig = sig_pivot.values[i, j]
        if np.isnan(val):
            continue
        star = ' *' if is_sig else ''
        text_color = 'white' if val > 1.3 or val < 0.85 else 'black'
        ax.text(j, i, f'{val:.2f}{star}', ha='center', va='center',
                color=text_color, fontsize=9, fontweight='bold')

cbar = fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
cbar.set_label('Fold Enrichment', fontsize=11)
plt.tight_layout()
plt.show()

---
## 7. Distance Profile: Boundary Analysis

The compartment-level test tells us *whether* an LR pair is enriched in a compartment,
but not *how* the enrichment varies with distance from the boundary. The **distance
profile** fills this gap by computing LR co-expression and spatial fold enrichment in
narrow distance bins across the tumor boundary.

We use `compute_distance_profile()` to compute:

- **Coexpression fraction** (L+R+ / L+): what fraction of ligand-positive cells also
  express the receptor, as a function of distance
- **Spatial fold enrichment**: uses a binary neighbor test as a quick descriptive
  summary (fraction of L+ cells with an R+ neighbor / expected). Note: Figure 5d in the
  paper uses the distance-weighted kernel fold enrichment (T/E[T], tau=7) for consistency
  with the CONSTELLATION test statistic.

We plot this for **CXCL13-CXCR5**, a chemokine axis involved in tertiary lymphoid
structure (TLS) formation, which shows the strongest enrichment in the Tissue compartment.

The boundary profile plot shows three rows:
1. **Top**: Coexpression fraction vs distance
2. **Middle**: Number of ligand-positive cells per bin (sample size)
3. **Bottom**: Spatial fold enrichment vs distance (y=1 = no enrichment)

In [ ]:
# Compute distance profile for CXCL13-CXCR5
profile = cst.compute_distance_profile(
    expr_dict=expr_dict,
    ligand='CXCL13',
    receptor='CXCR5',
    distance_values=distance_values,
    indices=knn_indices,
    bin_width=15,
    min_cells=100,
)

# Plot boundary profile — limit to ±200 µm around boundary, label zones
fig, axes = cst.plot_boundary_profile(
    profile,
    ligand='CXCL13',
    receptor='CXCR5',
    boundary_pos=0,
    interface_width=50,
    xlim=(-200, 300),
    zone_labels=True,
)
plt.show()

---
## 8. Save Results

Save the full results table as a CSV for downstream analysis or figure generation.

In [ ]:
results.to_csv(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH} ({len(results)} rows)')
print(f'  Significant: {len(sig)} interactions')
print(f'  Compartments: {results["compartment"].nunique()}')
print(f'  LR pairs tested: {results["lr_pair"].nunique()}')